# AI adoption, digital-skill diffusion and achievement gaps on a homophilous school network

Agent-based simulation code accompanying the manuscript. The core model
(`model.py`) simulates AI-tool adoption as a complex contagion on an SES-homophilous
student network, digital-skill diffusion, and latent achievement accumulation.
Experiments E0–E20 reproduce every table and figure in the paper.

## Design
All primary outcome comparisons are evaluated at a **common fixed horizon of
50 simulation periods** (`manuscript.py`). Cascade-stationarity time is reported
as a diagnostic only; scenarios are never compared at different elapsed times.
Paired contrasts use common random numbers (same seed → same population and
network for both arms), and every Monte-Carlo standard error uses the actual
replication count.

## Experiment map
| ID | Content |
|----|---------|
| E0 | Structural baseline (no AI) |
| E1, E1b, E1c | Homophily × access-gap phase sweep, channel decomposition, gap × initial-skill sweep |
| E2 | Adoption / skill / achievement trajectories |
| E3, E4, E15 | Policy comparison; targeting × budget; full intervention-unit sweep |
| E5, E8 | Global sensitivity (Morris screening → Sobol; confirmatory 7-parameter Sobol) |
| E6, E7 | Structural robustness (topology, contagion, scheduling, learning); MC convergence |
| E9, E16 | Finite-size / density robustness; fixed-class-size check |
| E10, E11, E19, E20 | Seed-placement-only targeting, saturation diagnostic, threshold sweeps, mean-field comparison |
| E12, E13, E14 | Scale-parameter, access-shape and classroom-sorting robustness |
| E17 | Three-group reference outcomes |
| E18 | Channel decomposition at a 20-period horizon |

## How to run
1. Run the cells in order. `runmode.py` defaults to `FAST_MODE = True`, which
   uses ~1/10 of the replications and is intended only to check that the
   pipeline executes end to end.
2. For manuscript-grade results, set `FAST_MODE = False` in `runmode.py`,
   restart the runtime, and run all cells. Package versions and run metadata
   are exported automatically alongside the results.

Outputs are written to `results/`, `results_extended/`, `results_robustness/`,
`round2_targeted_results/`, `final_theta_sweep_results/`, `figs/` and `figs_extended/`.

In [ ]:
!pip -q install SALib


In [ ]:
import numpy as np, random, os, sys, importlib.metadata as im
BASE_SEED = 20260901
def rng(offset=0):
    return np.random.default_rng(BASE_SEED + offset)
np.random.seed(BASE_SEED); random.seed(BASE_SEED)
print('base seed', BASE_SEED)
print('Python', sys.version.split()[0])
for pkg in ['numpy','scipy','pandas','networkx','matplotlib','SALib']:
    try:
        print(pkg, im.version(pkg))
    except Exception:
        pass


In [ ]:
%%writefile runmode.py
"""Run-mode switch shared by all experiment cells."""
FAST_MODE = True   # set False for manuscript-grade full run

def RR(R):
    """Replication count actually used: 1/10 of R (min 3) in FAST_MODE, else R."""
    return max(3, R // 10) if FAST_MODE else R

def banner():
    if FAST_MODE:
        print('*** FAST_MODE: pipeline test only - not for manuscript reporting ***')


In [ ]:
%%writefile model.py
"""Core model: AI adoption, digital-skill diffusion and learning on a homophilous network.
All dynamics are vectorized with numpy.
"""
import numpy as np
from dataclasses import dataclass, field, replace, asdict

GROUPS = ("L", "M", "H")
G_IDX = {"L": 0, "M": 1, "H": 2}

@dataclass
class Config:
    # population
    N: int = 500
    frac: tuple = (0.4, 0.4, 0.2)          # L, M, H shares
    C: int = 25                             # classrooms
    # network (degree-balanced block model)
    kbar: float = 8.0
    h: float = 0.6                          # homophily in [0,1)
    topology: str = "sbm"                  # 'sbm' | 'dcsbm'
    # classroom sorting
    rho: float = 0.5
    # physical access: population mean fixed, H-L gap varied
    pbar: float = 0.6
    dp: float = 0.5                         # p_H - p_L
    shape_mid: float = 0.615                # stylized p_M placement: p_M = p_L + shape_mid*dp
    # initial digital skill Beta(mu*kap,(1-mu)*kap)
    ds0: float = 0.3                        # mu_H - mu_L, mu_M = 0.5
    skill_conc: float = 8.0
    # adoption (complex contagion)
    theta0: float = 0.2
    lam: float = 0.5
    sigma0: float = 0.10                    # seed fraction among access-holders
    contagion: str = "complex"             # 'complex' | 'simple'
    beta_c: float = 0.08                    # per-contact prob (simple contagion variant)
    # skill dynamics
    eta: float = 0.05
    gamma_s: float = 0.10
    # learning (latent additive)
    beta_alpha: float = 0.010
    mu_alpha: float = 1.0
    sigma_u: float = 0.15                   # alpha_i = mu_alpha + u_i (random effect)
    beta_T: float = 0.005
    beta_AI: float = 0.012
    sigma_eps: float = 0.010
    A0: float = 0.0
    learning: str = "latent"               # 'latent' | 'saturating' (robustness)
    Amax: float = 2.0                       # saturating variant only
    # teacher support: Normal(0.6,0.2) clipped to [0.1,1]
    T_mu: float = 0.6
    T_sd: float = 0.2
    # scheduling / horizon
    sync: bool = True
    patience: int = 5
    T_post: int = 25
    T_max: int = 100
    # policy
    policy: str = "no_ai"
    budget: int = 60
    skill_boost: float = 0.2
    deseg_h: float = 0.3                    # h used by 'deseg' (applied before network generation)
    deseg_rho: float = 0.0

def access_probs(cfg):
    # Constant-population-mean access design:
    # frac_L*pL + frac_M*pM + frac_H*pH = pbar,
    # pM = pL + shape_mid*dp, pH = pL + dp.
    fL, fM, fH = cfg.frac
    pL = cfg.pbar - (fM * cfg.shape_mid + fH) * cfg.dp
    pM = pL + cfg.shape_mid * cfg.dp
    pH = pL + cfg.dp
    raw = np.array([pL, pM, pH], dtype=float)
    if np.all((raw >= 0.0) & (raw <= 1.0)):
        assert abs(np.dot(np.asarray(cfg.frac, float), raw) - cfg.pbar) < 1e-12
    return np.clip(raw, 0.0, 1.0)

def gen_network(cfg, ses, rng):
    """Degree-balanced block model: E[deg]=kbar for every SES group at any h.
    p_out = kbar*(1-h)/(N-1);  p_in,g = omega_g*kbar/(n_g-1),
    omega_g = omega0_g + h*(1-omega0_g), omega0_g=(n_g-1)/(N-1)."""
    N = cfg.N
    n = np.array([(ses == g).sum() for g in range(3)])
    omega0 = (n - 1) / (N - 1)
    omega = omega0 + cfg.h * (1 - omega0)
    p_in = omega * cfg.kbar / (n - 1)
    p_out = cfg.kbar * (1 - cfg.h) / (N - 1)
    P = np.full((3, 3), p_out)
    np.fill_diagonal(P, p_in)
    Pm = P[ses[:, None], ses[None, :]]
    if cfg.topology == "dcsbm":  # degree-corrected variant (E6): lognormal propensities, group-mean 1
        prop = rng.lognormal(mean=-0.125, sigma=0.5, size=N)
        for g in range(3):
            prop[ses == g] /= prop[ses == g].mean()
        Pm = np.clip(Pm * prop[:, None] * prop[None, :], 0, 1)
    U = rng.random((N, N))
    A = (U < Pm)
    A = np.triu(A, 1)
    A = A | A.T
    np.fill_diagonal(A, False)
    return A

def ses_assortativity(A, ses):
    """Newman categorical assortativity for SES on edges."""
    idx = np.array(np.nonzero(np.triu(A, 1)))
    if idx.shape[1] == 0:
        return np.nan
    gi, gj = ses[idx[0]], ses[idx[1]]
    e = np.zeros((3, 3))
    for a, b in zip(gi, gj):
        e[a, b] += 1; e[b, a] += 1
    e /= e.sum()
    ai = e.sum(1)
    tr = np.trace(e)
    return (tr - (ai ** 2).sum()) / (1 - (ai ** 2).sum())

def assign_classrooms(cfg, ses, rng):
    N, C = cfg.N, cfg.C
    size = N // C
    slots = np.repeat(np.arange(C), size)
    order = np.argsort(ses, kind="stable")          # SES-sorted stream
    n_sorted = int(round(cfg.rho * N))
    # choose which students are sorted-assigned (take a random subset of the sorted stream positions)
    pick = rng.choice(N, size=n_sorted, replace=False)
    is_sorted = np.zeros(N, bool); is_sorted[pick] = True
    room = np.empty(N, int)
    # sorted students occupy slots in SES order; the rest fill remaining slots randomly
    sorted_stream = order[is_sorted[order]]
    room[sorted_stream] = slots[:n_sorted]
    rest = order[~is_sorted[order]]
    rest_slots = slots[n_sorted:].copy(); rng.shuffle(rest_slots)
    room[rest] = rest_slots
    return room

def build_world(cfg, seed):
    """Everything drawn before dynamics. Structural policies act on cfg BEFORE generation."""
    cfg = apply_structural_policy(cfg)
    rng = np.random.default_rng(seed)
    N = cfg.N
    n = [int(round(f * N)) for f in cfg.frac]; n[2] = N - n[0] - n[1]
    ses = np.concatenate([np.full(k, g) for g, k in enumerate(n)])
    rng.shuffle(ses)
    p = access_probs(cfg)
    c = (rng.random(N) < p[ses]).astype(np.int8)
    muL, muH = 0.5 - cfg.ds0 / 2, 0.5 + cfg.ds0 / 2
    mus = np.array([muL, 0.5, muH]); kap = cfg.skill_conc
    s = rng.beta(np.maximum(mus[ses] * kap, 1e-3), np.maximum((1 - mus[ses]) * kap, 1e-3))
    alpha = cfg.mu_alpha + rng.normal(0, cfg.sigma_u, N)   # random-effect formulation
    A = gen_network(cfg, ses, rng)
    deg = A.sum(1)
    room = assign_classrooms(cfg, ses, rng)
    Tc = np.clip(rng.normal(cfg.T_mu, cfg.T_sd, cfg.C), 0.1, 1.0)
    T = Tc[room]
    theta = cfg.theta0 * (1 - cfg.lam * T)
    a = np.zeros(N, np.int8)
    Aach = np.full(N, cfg.A0, float)
    world = dict(cfg=cfg, rng=rng, ses=ses, c=c, s=s, alpha=alpha, adj=A, deg=deg,
                 room=room, T=T, theta=theta, a=a, ach=Aach, ai_on=True)
    apply_agent_policy(world)
    # seed adoption uniformly among access-holders (unless a targeting policy already seeded)
    if world["a"].sum() == 0 and world["ai_on"]:
        holders = np.flatnonzero(world["c"] == 1)
        k = int(round(cfg.sigma0 * len(holders)))
        if k > 0:
            world["a"][rng.choice(holders, size=k, replace=False)] = 1
    return world

# ---------------- policies ----------------
def apply_structural_policy(cfg):
    if cfg.policy == "deseg":
        return replace(cfg, h=cfg.deseg_h, rho=cfg.deseg_rho)
    return cfg

def _seed_and_equip(world, targets):
    world["c"][targets] = 1
    world["a"][targets] = 1

def cross_frac(world):
    A, ses = world["adj"], world["ses"]
    deg = np.maximum(world["deg"], 1)
    same = (ses[:, None] == ses[None, :])
    return (A & ~same).sum(1) / deg

def betweenness(world, rng):
    import networkx as nx
    G = nx.from_numpy_array(world["adj"])
    return np.array(list(nx.betweenness_centrality(G, k=64, seed=int(rng.integers(1e9))).values()))

def apply_agent_policy(world):
    cfg, rng = world["cfg"], world["rng"]
    pol, B = cfg.policy, cfg.budget
    L = np.flatnonzero(world["ses"] == 0)
    if pol == "no_ai":
        world["ai_on"] = False
    elif pol in ("universal", "deseg"):
        pass                                            # availability only: physical access c_i unchanged
    elif pol == "access_full":
        world["c"][:] = 1                               # universal physical-access equalization
    elif pol == "access_budget":
        no_acc = L[world["c"][L] == 0]
        k = min(B, len(no_acc))
        if k: world["c"][rng.choice(no_acc, size=k, replace=False)] = 1
    elif pol == "skills":
        order = L[np.argsort(world["s"][L])][:B]
        world["s"][order] = np.clip(world["s"][order] + cfg.skill_boost, 0, 1)
    elif pol == "target_random":
        t = rng.choice(L, size=min(B, len(L)), replace=False); _seed_and_equip(world, t)
    elif pol == "target_lowskill":
        t = L[np.argsort(world["s"][L])][:B]; _seed_and_equip(world, t)
    elif pol == "target_degree":
        t = L[np.argsort(-world["deg"][L])][:B]; _seed_and_equip(world, t)
    elif pol == "target_betweenness":
        bc = betweenness(world, rng); t = L[np.argsort(-bc[L])][:B]; _seed_and_equip(world, t)
    elif pol == "target_bridge":
        cf = cross_frac(world); elig = L[world["deg"][L] >= 3]
        t = elig[np.argsort(-cf[elig])][:B]; _seed_and_equip(world, t)
    elif pol == "combined":
        b = B // 3
        no_acc = L[world["c"][L] == 0]
        if len(no_acc): world["c"][rng.choice(no_acc, size=min(b, len(no_acc)), replace=False)] = 1
        order = L[np.argsort(world["s"][L])][:b]
        world["s"][order] = np.clip(world["s"][order] + cfg.skill_boost, 0, 1)
        bc = betweenness(world, rng); t = L[np.argsort(-bc[L])][:B - 2 * b]; _seed_and_equip(world, t)
    else:
        raise ValueError(pol)

# ---------------- dynamics ----------------
def step(world):
    cfg = world["cfg"]; A = world["adj"]; rng = world["rng"]
    a, s = world["a"], world["s"]
    deg = np.maximum(world["deg"], 1)
    if world["ai_on"]:
        na = A @ a
        if cfg.contagion == "complex":
            w = na / deg
            new = (world["c"] == 1) & (w >= world["theta"])
        else:                                            # simple contagion (E6)
            pr = 1 - (1 - cfg.beta_c) ** na
            new = (world["c"] == 1) & (rng.random(cfg.N) < pr)
        if cfg.sync:
            a_next = np.where(new, 1, a).astype(np.int8)
        else:                                            # asynchronous: random half updates (E6)
            upd = rng.random(cfg.N) < 0.5
            a_next = a.copy(); a_next[upd & new] = 1
    else:
        a_next = a
    # skills (S1): uses current a
    up = A & (s[None, :] > s[:, None])
    cnt = up.sum(1)
    sbar = np.where(cnt > 0, (up * s[None, :]).sum(1) / np.maximum(cnt, 1), s)
    gain = cfg.eta * a * (1 - s) + cfg.gamma_s * np.maximum(sbar - s, 0)
    s_next = s + gain
    # learning (L1)
    ai_term = cfg.beta_AI * a * s if world["ai_on"] else 0.0
    inc = cfg.beta_alpha * world["alpha"] + cfg.beta_T * world["T"] + ai_term \
          + rng.normal(0, cfg.sigma_eps, cfg.N)
    if cfg.learning == "latent":
        ach_next = world["ach"] + inc
    else:                                                # saturating robustness variant
        ach_next = world["ach"] + inc * (1 - world["ach"] / cfg.Amax)
    changed = int((a_next != a).sum())
    world["a"], world["s"], world["ach"] = a_next, s_next, ach_next
    return changed

def run_simulation(cfg, seed, trajectories=False):
    world = build_world(cfg, seed)
    cfg = world["cfg"]
    ses = world["ses"]
    quiet, t, tstar = 0, 0, None
    traj = []
    def snap():
        return [t] + [world["a"][ses == g].mean() for g in range(3)] \
                   + [world["s"][ses == g].mean() for g in range(3)] \
                   + [world["ach"][ses == g].mean() for g in range(3)]
    while t < cfg.T_max:
        if trajectories: traj.append(snap())
        ch = step(world)
        t += 1
        # validation each step
        assert world["s"].min() >= -1e-12 and world["s"].max() <= 1 + 1e-12, "skill bounds violated"
        assert np.isfinite(world["ach"]).all(), "achievement NaN/inf"
        quiet = quiet + 1 if ch == 0 else 0
        if tstar is None and quiet >= cfg.patience:
            tstar = t - cfg.patience
        if tstar is not None and t >= tstar + cfg.T_post:
            break
    if trajectories: traj.append(snap())
    if cfg.contagion == "complex" and cfg.sync:
        pass  # monotonicity guaranteed by construction (a_next >= a)
    out = metrics(world)
    out.update(t_star=tstar if tstar is not None else cfg.T_max, T_end=t,
               assort=ses_assortativity(world["adj"], ses),
               deg_all=float(world["deg"].mean()),
               **{f"deg_{g}": float(world["deg"][ses == G_IDX[g]].mean()) for g in GROUPS})
    return (out, np.array(traj)) if trajectories else out

def metrics(world):
    from scipy.stats import wasserstein_distance
    ses, ach = world["ses"], world["ach"]
    a, s, c = world["a"], world["s"], world["c"]
    m = {}
    for g in GROUPS:
        i = ses == G_IDX[g]
        m[f"ach_{g}"] = ach[i].mean(); m[f"x_{g}"] = a[i].mean()
        m[f"s_{g}"] = s[i].mean(); m[f"as_{g}"] = (a[i] * s[i]).mean()
        m[f"acc_{g}"] = c[i].mean()
    nH, nL = (ses == 2).sum(), (ses == 0).sum()
    vH, vL = ach[ses == 2].var(ddof=1), ach[ses == 0].var(ddof=1)
    sp = np.sqrt(((nH - 1) * vH + (nL - 1) * vL) / (nH + nL - 2))
    m["G_HL"] = m["ach_H"] - m["ach_L"]
    m["d_HL"] = m["G_HL"] / sp if sp > 0 else 0.0
    m["sd"] = ach.std(ddof=1)
    m["q90_10"] = np.quantile(ach, 0.9) - np.quantile(ach, 0.1)
    gm = np.array([ach[ses == g].mean() for g in range(3)])
    ns = np.array([(ses == g).sum() for g in range(3)])
    ssb = (ns * (gm - ach.mean()) ** 2).sum()
    m["bg_share"] = ssb / ((ach - ach.mean()) ** 2).sum()
    m["wass_HL"] = wasserstein_distance(ach[ses == 2], ach[ses == 0])
    m["x_all"] = a.mean(); m["as_gap"] = m["as_H"] - m["as_L"]
    m["mean_ach"] = ach.mean()
    return m

def run_mc(cfg, R, base_seed=0):
    import pandas as pd
    rows = [run_simulation(cfg, base_seed + 1000 * r) for r in range(R)]
    return pd.DataFrame(rows)

def paired_contrast(cfg_a, cfg_b, R, base_seed=0, cols=("d_HL", "G_HL", "mean_ach", "sd", "bg_share", "x_L", "x_H", "as_gap")):
    """Common random numbers: same seed -> same population/network draws for both configs."""
    import pandas as pd
    rows = []
    for r in range(R):
        sd = base_seed + 1000 * r
        oa, ob = run_simulation(cfg_a, sd), run_simulation(cfg_b, sd)
        rows.append({f"{k}_a": oa[k] for k in cols} | {f"{k}_b": ob[k] for k in cols} |
                    {f"d_{k}": ob[k] - oa[k] for k in cols})
    return pd.DataFrame(rows)

def mean_field_boundary(cfg, h_grid, dp_grid, iters=200):
    """Two-block-family mean-field: predicted adoption fixed point per SES group and
    the implied effective-use differential under universal availability. Derived calculation,
    compared with (not fitted to) simulation."""
    from scipy.stats import binom
    N = cfg.N; n = np.asarray(cfg.frac, float) * N
    thbar = cfg.theta0 * (1 - cfg.lam * cfg.T_mu)
    k = int(round(cfg.kbar))
    out = np.zeros((len(h_grid), len(dp_grid)))
    for ih, h in enumerate(h_grid):
        omega0 = (n - 1) / (N - 1); omega = omega0 + h * (1 - omega0)
        # neighbour-composition matrix m[g,g']: share of g's neighbours in g'
        M = np.zeros((3, 3))
        for g in range(3):
            M[g, g] = omega[g]
            others = [gg for gg in range(3) if gg != g]
            tot = sum(n[gg] for gg in others)
            for gg in others:
                M[g, gg] = (1 - omega[g]) * n[gg] / tot
        for idp, dp in enumerate(dp_grid):
            p = access_probs(replace(cfg, dp=dp))
            x = p * cfg.sigma0
            for _ in range(iters):
                w = M @ x
                # P(Binom(k, w) >= ceil(theta*k)) smoothed threshold response
                thr = int(np.ceil(thbar * k))
                adopt_pr = 1 - binom.cdf(thr - 1, k, np.clip(w, 0, 1))
                x = p * (cfg.sigma0 + (1 - cfg.sigma0) * adopt_pr)
            out[ih, idp] = x[2] - x[0]     # adoption differential H - L (drives as-gap sign)
    return out


In [ ]:
"""Automated validation and policy-integrity tests. Any failure aborts the run."""
import numpy as np
from dataclasses import replace
from model import *

def world_state(pol, seed=11, **kw):
    w = build_world(replace(Config(policy=pol), **kw), seed)
    return w

def main():
    rep = []
    # -- access-gap parameterization preserves the population mean pbar
    for dp in [0.0, 0.32, 0.64]:
        cfg = Config(dp=dp)
        p = access_probs(cfg)
        assert abs(np.dot(np.asarray(cfg.frac), p) - cfg.pbar) < 1e-12, (
            f"mean access not preserved at dp={dp}: p={p}, mean={np.dot(cfg.frac,p)}"
        )
    rep.append(("constant mean access across dp", True))
    # -- network: degree balance across SES at several h (tolerance 5%)
    for h in [0.0, 0.3, 0.6, 0.9]:
        outs = [run_simulation(Config(policy="no_ai", h=h, T_max=1, T_post=0, patience=1), s) for s in range(5)]
        for g in ("L","M","H"):
            dev = abs(np.mean([o[f"deg_{g}"] for o in outs]) - np.mean([o["deg_all"] for o in outs])) / 8.0
            assert dev < 0.05, f"degree imbalance g={g} h={h} dev={dev:.3f}"
        rep.append((f"degree balance h={h}", True))
    # -- assortativity responds to h
    r0 = np.mean([ses_assortativity(world_state("no_ai", s, h=0.0)["adj"], world_state("no_ai", s, h=0.0)["ses"]) for s in range(3)])
    r9 = np.mean([ses_assortativity(world_state("no_ai", s, h=0.9)["adj"], world_state("no_ai", s, h=0.9)["ses"]) for s in range(3)])
    assert r9 > r0 + 0.5, f"assortativity does not respond to h: {r0:.2f} vs {r9:.2f}"
    rep.append((f"assortativity responds to h ({r0:.2f}->{r9:.2f})", True))
    # -- universal availability must not change physical access c
    w_no, w_un = world_state("no_ai"), world_state("universal")
    assert np.array_equal(w_no["c"], w_un["c"]), "universal availability altered c_i"
    rep.append(("universal availability leaves c_i untouched", True))
    # -- access_full sets c=1
    assert world_state("access_full")["c"].min() == 1; rep.append(("access_full sets c_i=1", True))
    # -- budgets: each targeted policy touches exactly B agents (or as many as eligible)
    for pol in ["target_random","target_lowskill","target_degree","target_betweenness","target_bridge"]:
        w = world_state(pol); n_seed = int(w["a"].sum())
        assert n_seed == w["cfg"].budget, f"{pol} budget mismatch: {n_seed}"
        assert set(np.flatnonzero(w["a"])) <= set(np.flatnonzero(w["ses"]==0)), f"{pol} targeted non-L agents"
    rep.append(("targeting budgets equal & low-SES only", True))
    # -- centrality targeting picks expected agents
    w = world_state("target_degree")
    L = np.flatnonzero(w["ses"]==0); chosen = np.flatnonzero(w["a"]==1)
    thr = np.sort(w["deg"][L])[::-1][w["cfg"].budget-1]
    assert w["deg"][chosen].min() >= thr, "degree targeting selection wrong"
    rep.append(("degree targeting selects top-degree low-SES", True))
    # -- desegregation modifies network structure before generation
    r_ref = ses_assortativity(world_state("universal", 5)["adj"], world_state("universal", 5)["ses"])
    r_des = ses_assortativity(world_state("deseg", 5)["adj"], world_state("deseg", 5)["ses"])
    assert r_des < r_ref - 0.1, f"deseg did not reduce assortativity ({r_ref:.2f}->{r_des:.2f})"
    rep.append((f"deseg reduces realized assortativity ({r_ref:.2f}->{r_des:.2f})", True))
    # -- distinct policies produce distinct agent trajectories under a fixed seed (inert-branch detector)
    pols = ["universal","access_full","access_budget","skills","target_random","target_betweenness","deseg","combined"]
    sigs = {}
    for p in pols:
        o = run_simulation(replace(Config(policy=p)), 123)
        sigs[p] = (round(o["mean_ach"],6), round(o["d_HL"],6), round(o["x_L"],6))
    vals = list(sigs.values())
    assert len(set(vals)) == len(vals), f"POTENTIAL INERT POLICY BRANCH: {sigs}"
    rep.append(("all policies yield distinct trajectories (no inert branch)", True))
    # -- monotone adoption & skill bounds & finite achievement (checked in-run by assertions) on a stress run
    run_simulation(Config(policy="universal", h=0.95, dp=0.64, theta0=0.35), 77)
    rep.append(("in-run assertions (bounds, finiteness) pass under stress config", True))
    # -- seeds unique / reproducibility: same seed same result, diff seed diff result
    o1, o2 = run_simulation(Config(policy="universal"), 42), run_simulation(Config(policy="universal"), 42)
    o3 = run_simulation(Config(policy="universal"), 43)
    assert o1["mean_ach"] == o2["mean_ach"] and o1["mean_ach"] != o3["mean_ach"]
    rep.append(("bitwise reproducibility under fixed seed", True))
    print("VALIDATION REPORT"); [print(f"  [{'PASS' if s else 'FAIL'}] {n}") for n, s in rep]
    print(f"{sum(s for _,s in rep)}/{len(rep)} checks passed")

main()


## Common manuscript horizon

In [ ]:
%%writefile manuscript.py
"""Primary manuscript configuration: a common fixed evaluation horizon."""
from model import Config

PRIMARY_HORIZON = 50

def PC(**kwargs):
    """Config evaluated at exactly PRIMARY_HORIZON periods (no early stopping)."""
    kwargs = dict(kwargs)
    kwargs["T_max"] = PRIMARY_HORIZON
    kwargs["T_post"] = PRIMARY_HORIZON
    kwargs["patience"] = PRIMARY_HORIZON + 1
    return Config(**kwargs)


## E0 structural baseline + E1 phase sweep (with mean-field prediction)

In [ ]:
import os, time
import numpy as np, pandas as pd
from dataclasses import replace
from model import *
from manuscript import PC, PRIMARY_HORIZON
from runmode import FAST_MODE, RR, banner
banner()
os.makedirs("results", exist_ok=True); os.makedirs("figs", exist_ok=True)

t0=time.time()
# ---------- E0: structural baseline (no AI) ----------
df0 = run_mc(PC(policy="no_ai"), R=RR(200), base_seed=100)
df0.to_csv("results/e0_baseline.csv", index=False)
s = df0[["d_HL","G_HL","sd","bg_share","mean_ach","assort","deg_L","deg_M","deg_H","t_star"]].agg(["mean","std"])
s.to_csv("results/e0_summary.csv")
print("E0 done", f"{time.time()-t0:.0f}s")
print(s.round(4).to_string())
mcse = df0["d_HL"].std()/np.sqrt(len(df0))
print(f"E0 d_HL = {df0['d_HL'].mean():.4f} (MC-SE {mcse:.4f}); 95% CI [{df0['d_HL'].mean()-1.96*mcse:.4f},{df0['d_HL'].mean()+1.96*mcse:.4f}]")

# ---------- E1: phase sweep h x dp (universal availability vs no AI, CRN-paired) ----------
H = np.round(np.linspace(0, 0.9, 9), 4); DP = np.round(np.linspace(0, 0.64, 9), 4); R1 = RR(30)
rows=[]
for h in H:
    for dp in DP:
        ca = PC(policy="no_ai", h=h, dp=dp); cb = PC(policy="universal", h=h, dp=dp)
        d = paired_contrast(ca, cb, R=R1, base_seed=int(20000+1e4*h+1e2*dp))
        dd = d["d_d_HL"]; se = dd.std()/np.sqrt(R1)
        rows.append(dict(h=h, dp=dp, dd_mean=dd.mean(), dd_se=se,
                         lo=dd.mean()-1.96*se, hi=dd.mean()+1.96*se,
                         dG_mean=d["d_G_HL"].mean(), dG_se=d["d_G_HL"].std()/np.sqrt(R1),
                         dmean_ach=d["d_mean_ach"].mean(),
                         xL=d["x_L_b"].mean(), xH=d["x_H_b"].mean(),
                         as_gap=d["as_gap_b"].mean(), R=R1))
e1 = pd.DataFrame(rows); e1.to_csv("results/e1_phase.csv", index=False)
print("E1 done", f"{time.time()-t0:.0f}s; cells={len(e1)} (expect 81)")
assert len(e1)==81, "phase sweep cell count wrong"
# mean-field prediction on same grid
mf = mean_field_boundary(PC(), H, DP)
pd.DataFrame(mf, index=H, columns=DP).to_csv("results/e1_meanfield.csv")
print("corner values dd:", e1[(e1.h==0)&(e1.dp==0)]["dd_mean"].iloc[0].round(3),
      e1[(e1.h==0.9)&(e1.dp==0.64)]["dd_mean"].iloc[0].round(3))
print("cells CI includes 0:", int(((e1.lo<0)&(e1.hi>0)).sum()), "/81; negative-mean cells:", int((e1.dd_mean<0).sum()))


## E1b channel decomposition, E1c secondary sweep, E2 diffusion

In [ ]:
import os, time
import numpy as np, pandas as pd
from dataclasses import replace
from model import *
from manuscript import PC, PRIMARY_HORIZON
from runmode import FAST_MODE, RR, banner
banner()
os.makedirs("results", exist_ok=True); os.makedirs("figs", exist_ok=True)
t0=time.time()
# ---------- E1b: channel decomposition (paired vs no-AI, CRN) ----------
configs = {
 "all_channels":        dict(dp=0.5, ds0=0.3, h=0.6),
 "equal_access":        dict(dp=0.0, ds0=0.3, h=0.6),
 "equal_skills":        dict(dp=0.5, ds0=0.0, h=0.6),
 "no_homophily":        dict(dp=0.5, ds0=0.3, h=0.0),
 "access_only":         dict(dp=0.5, ds0=0.0, h=0.0),
 "skills_only":         dict(dp=0.0, ds0=0.3, h=0.0),
 "no_inequality":       dict(dp=0.0, ds0=0.0, h=0.6),
 "pure_null":           dict(dp=0.0, ds0=0.0, h=0.0),
}
rows=[]
seed_offsets = {name: i for i, name in enumerate(configs)}
for name, kw in configs.items():
    d = paired_contrast(replace(PC(policy="no_ai"), **kw),
                        replace(PC(policy="universal"), **kw),
                        R=RR(100), base_seed=31000 + 100 * seed_offsets[name])
    dd=d["d_d_HL"]; se=dd.std()/np.sqrt(len(dd))
    rows.append(dict(config=name, **kw, dd=dd.mean(), se=se, lo=dd.mean()-1.96*se, hi=dd.mean()+1.96*se,
                     as_gap=d["as_gap_b"].mean(), xL=d["x_L_b"].mean(), xH=d["x_H_b"].mean(),
                     dmean=d["d_mean_ach"].mean()))
e1b=pd.DataFrame(rows); e1b.to_csv("results/e1b_channels.csv", index=False)
print(e1b[["config","dd","se","as_gap","xL","xH","dmean"]].round(4).to_string(index=False)); print(f"{time.time()-t0:.0f}s")

# ---------- E1c: secondary sweep dp x ds0 at h=0.6 (locate the null regime) ----------
rows=[]
for dp in np.round(np.linspace(0,0.64,5),3):
    for ds0 in np.round(np.linspace(0,0.4,5),3):
        d = paired_contrast(PC(policy="no_ai",dp=dp,ds0=ds0), PC(policy="universal",dp=dp,ds0=ds0), R=RR(30), base_seed=int(40000+1e3*dp+1e2*ds0))
        dd=d["d_d_HL"]; se=dd.std(ddof=1)/np.sqrt(len(dd))
        rows.append(dict(dp=dp, ds0=ds0, dd=dd.mean(), se=se, lo=dd.mean()-1.96*se, hi=dd.mean()+1.96*se))
e1c=pd.DataFrame(rows); e1c.to_csv("results/e1c_dp_ds0.csv", index=False)
print("E1c cells with CI incl 0:", int(((e1c.lo<0)&(e1c.hi>0)).sum()), "/25", f"{time.time()-t0:.0f}s")

# ---------- E2: diffusion trajectories h x gamma_s ----------
traj_rows=[]; summ=[]
for h in [0.0,0.3,0.6,0.9]:
    for gs in [0.0,0.1,0.2]:
        cfg = Config(policy="universal", h=h, gamma_s=gs, T_max=60, T_post=60, patience=999)
        TT=[]
        for r in range(RR(40)):
            out, tr = run_simulation(cfg, 50000+r, trajectories=True)
            TT.append(tr)
        n = min(t.shape[0] for t in TT)
        M = np.mean([t[:n] for t in TT], axis=0)
        for row in M:
            traj_rows.append(dict(h=h, gamma_s=gs, t=row[0], xL=row[1], xM=row[2], xH=row[3],
                                  sL=row[4], sM=row[5], sH=row[6], aL=row[7], aM=row[8], aH=row[9]))
        xLf,xHf = M[-1,1],M[-1,3]
        t50L = next((int(r[0]) for r in M if r[1]>=0.5*max(xLf,1e-9)), None) if xLf>0 else None
        t50H = next((int(r[0]) for r in M if r[3]>=0.5*xHf), None)
        summ.append(dict(h=h,gamma_s=gs,R=len(TT),xL_final=xLf,xH_final=xHf,t50L=t50L,t50H=t50H))
pd.DataFrame(traj_rows).to_csv("results/e2_trajectories.csv", index=False)
e2=pd.DataFrame(summ); e2.to_csv("results/e2_summary.csv", index=False)
print(e2.round(3).to_string(index=False)); print(f"{time.time()-t0:.0f}s total")


## E3 policy comparison

In [ ]:
import os, time
import numpy as np, pandas as pd
from dataclasses import replace
from model import *
from manuscript import PC, PRIMARY_HORIZON
from runmode import FAST_MODE, RR, banner
banner()
os.makedirs("results", exist_ok=True); os.makedirs("figs", exist_ok=True)
t0=time.time()
POLS = ["universal","access_full","access_budget","skills","target_random","target_lowskill",
        "target_degree","target_betweenness","target_bridge","deseg","combined"]
# ---------- E3: policy comparison, all vs no_ai (CRN), budget=60 where budgeted ----------
rows=[]; raw=[]
for pol in POLS:
    d = paired_contrast(PC(policy="no_ai"), PC(policy=pol), R=RR(100), base_seed=60000)
    dd, dm = d["d_d_HL"], d["d_mean_ach"]
    rows.append(dict(policy=pol,
                     dd=dd.mean(), dd_se=dd.std(ddof=1)/np.sqrt(len(dd)),
                     dd_lo=dd.mean()-1.96*dd.std(ddof=1)/np.sqrt(len(dd)),
                     dd_hi=dd.mean()+1.96*dd.std(ddof=1)/np.sqrt(len(dd)),
                     dmean=dm.mean(), dmean_se=dm.std(ddof=1)/np.sqrt(len(dm)),
                     xL=d["x_L_b"].mean(), xH=d["x_H_b"].mean(), as_gap=d["as_gap_b"].mean(),
                     bg_share=d["bg_share_b"].mean()))
    raw.append(d.assign(policy=pol))
e3=pd.DataFrame(rows); e3.to_csv("results/e3_policies.csv", index=False)
pd.concat(raw).to_csv("results/e3_raw.csv", index=False)
print(e3[["policy","dd","dd_se","dmean","xL","xH"]].round(4).to_string(index=False)); print(f"{time.time()-t0:.0f}s")


## E4 targeting × budget

In [ ]:
import os, time
import numpy as np, pandas as pd
from dataclasses import replace
from model import *
from manuscript import PC, PRIMARY_HORIZON
from runmode import FAST_MODE, RR, banner
banner()
os.makedirs("results", exist_ok=True); os.makedirs("figs", exist_ok=True)
t0=time.time()
# ---------- E4: targeting strategies x budgets ----------
rows=[]
for B in [15,30,60,90]:
    for pol in ["target_random","target_lowskill","target_degree","target_betweenness","target_bridge"]:
        d = paired_contrast(PC(policy="no_ai"), replace(PC(policy=pol), budget=B), R=RR(80), base_seed=70000+B)
        dd=d["d_d_HL"]
        rows.append(dict(policy=pol, B=B, dd=dd.mean(), se=dd.std(ddof=1)/np.sqrt(len(dd)),
                         xL=d["x_L_b"].mean(), dmean=d["d_mean_ach"].mean()))
e4=pd.DataFrame(rows); e4.to_csv("results/e4_targeting.csv", index=False)
print(e4.pivot(index="policy", columns="B", values="dd").round(3).to_string())
print(e4.pivot(index="policy", columns="B", values="xL").round(3).to_string()); print(f"{time.time()-t0:.0f}s")


## E5a global sensitivity: Morris screening

In [ ]:
import os, time
import numpy as np, pandas as pd
from dataclasses import replace
from model import *
from manuscript import PC, PRIMARY_HORIZON
from runmode import FAST_MODE, RR, banner
banner()
os.makedirs("results", exist_ok=True); os.makedirs("figs", exist_ok=True)
from SALib.sample import morris as msample, saltelli
from SALib.analyze import morris as manalyze, sobol
t0=time.time()
PROB = {"num_vars":10,
 "names":["h","rho","dp","ds0","theta0","lam","gamma_s","eta","beta_AI","sigma0"],
 "bounds":[[0,0.9],[0,1],[0,0.64],[0,0.4],[0.1,0.3],[0,0.8],[0,0.2],[0.01,0.1],[0.004,0.02],[0.02,0.2]]}
INNER = 2 if FAST_MODE else 5
def resp(theta, base_seed):
    kw = dict(zip(PROB["names"], theta))
    vals=[run_simulation(replace(PC(policy="universal"), **kw), base_seed+13*r)["d_HL"] for r in range(INNER)]
    return np.mean(vals), np.var(vals, ddof=1)
# ---------- Morris screening ----------
X = msample.sample(PROB, N=RR(20), num_levels=4, seed=1)
Y = np.empty(len(X)); W = np.empty(len(X))
for i,th in enumerate(X):
    Y[i], W[i] = resp(th, 80000+i)
mres = manalyze.analyze(PROB, X, Y, num_levels=4, seed=1)
mdf = pd.DataFrame({k: mres[k] for k in ["names","mu_star","mu_star_conf","sigma"]})
mdf["within_theta_var_mean"] = W.mean()
mdf.sort_values("mu_star", ascending=False).to_csv("results/e5_morris.csv", index=False)
print(mdf.sort_values("mu_star", ascending=False).round(3).to_string(index=False))
print(f"Morris: {len(X)} vectors x {INNER} inner, {time.time()-t0:.0f}s; mean within-theta MC var {W.mean():.4f}")
top = list(mdf.sort_values("mu_star", ascending=False)["names"].head(5))
print("retained for Sobol:", top)
# Sobol analysis is performed in the following dedicated cell using the retained names.


## E5b global sensitivity: Sobol analysis on Morris-retained parameters

In [ ]:
import os, time
import numpy as np, pandas as pd
from dataclasses import replace
from model import *
from manuscript import PC, PRIMARY_HORIZON
from runmode import FAST_MODE, RR, banner
banner()
os.makedirs("results", exist_ok=True); os.makedirs("figs", exist_ok=True)
from SALib.sample import sobol as ssample
from SALib.analyze import sobol
t0=time.time()

# Retain the five highest-mu* parameters from the preceding Morris screen.
mdf = pd.read_csv("results/e5_morris.csv").sort_values("mu_star", ascending=False)
top = list(mdf["names"].head(5))
all_bounds = {
    "h":[0,0.9], "rho":[0,1], "dp":[0,0.64], "ds0":[0,0.4],
    "theta0":[0.1,0.3], "lam":[0,0.8], "gamma_s":[0,0.2],
    "eta":[0.01,0.1], "beta_AI":[0.004,0.02], "sigma0":[0.02,0.2]
}
P2 = {"num_vars":len(top), "names":top, "bounds":[all_bounds[n] for n in top]}
INNER = 2 if FAST_MODE else 8

def resp_subset(names, theta, base_seed):
    kw = dict(zip(names, theta))
    vals = [
        run_simulation(replace(PC(policy="universal"), **kw),
                       base_seed + 13*r)["d_HL"]
        for r in range(INNER)
    ]
    return np.mean(vals), np.var(vals, ddof=1)

X2 = ssample.sample(P2, (8 if FAST_MODE else 128), calc_second_order=False, seed=1)
Y2 = np.empty(len(X2)); W2 = np.empty(len(X2))
for i, th in enumerate(X2):
    Y2[i], W2[i] = resp_subset(top, th, 90000+i)

sres = sobol.analyze(P2, Y2, calc_second_order=False, seed=1)
sdf = pd.DataFrame({
    "name":top,
    "S1":sres["S1"], "S1_conf":sres["S1_conf"],
    "ST":sres["ST"], "ST_conf":sres["ST_conf"]
})
sdf["interaction"] = sdf.ST - sdf.S1
sdf.to_csv("results/e5_sobol.csv", index=False)
np.savetxt("results/e5_sobol_Y.csv", Y2, delimiter=",")
noise_share = (W2.mean()/INNER)/Y2.var()
print("retained parameters:", top)
print(sdf.round(3).to_string(index=False))
print(f"noise share={noise_share:.3f}; {len(X2)} vectors x {INNER} inner; {time.time()-t0:.0f}s")


## E6 structural robustness + E7 Monte Carlo convergence

In [ ]:
import os, time
import numpy as np, pandas as pd
from dataclasses import replace
from model import *
from manuscript import PC, PRIMARY_HORIZON
from runmode import FAST_MODE, RR, banner
banner()
os.makedirs("results", exist_ok=True); os.makedirs("figs", exist_ok=True)
t0=time.time()
# ---------- E6: structural robustness on mini-grid (paired) ----------
variants = {"reference": {}, "dcsbm": {"topology":"dcsbm"}, "simple_contagion": {"contagion":"simple"},
            "async": {"sync":False}, "saturating": {"learning":"saturating"}}
rows=[]
for vname, vkw in variants.items():
    for h in [0.0,0.45,0.9]:
        for dp in [0.0,0.32,0.64]:
            ca = replace(PC(policy="no_ai", h=h, dp=dp), **vkw)
            cb = replace(PC(policy="universal", h=h, dp=dp), **vkw)
            d = paired_contrast(ca, cb, R=RR(20), base_seed=int(95000+1e3*h+1e2*dp))
            dd = d["d_d_HL"]; se = dd.std(ddof=1)/np.sqrt(len(dd))
            rows.append(dict(variant=vname, h=h, dp=dp, dd=dd.mean(), se=se,
                             lo=dd.mean()-1.96*se, hi=dd.mean()+1.96*se, xL=d["x_L_b"].mean()))
e6=pd.DataFrame(rows); e6.to_csv("results/e6_robustness.csv", index=False)
piv = e6.pivot_table(index=["variant"], columns=["h","dp"], values="dd").round(2)
print(piv.to_string())
print("qualitative pattern holds (dd>0 where dp>0):", dict(e6[e6.dp>0].groupby("variant").apply(lambda g:(g.lo>0).all())))
print(f"{time.time()-t0:.0f}s")
# ---------- E7: Monte Carlo convergence at reference ----------
d = paired_contrast(PC(policy="no_ai"), PC(policy="universal"), R=RR(400), base_seed=99000)
d.to_csv("results/e7_raw.csv", index=False)
rows=[]
for R in [25,50,100,200,400]:
    if R > len(d): break
    x = d["d_d_HL"].iloc[:R]
    rows.append(dict(R=R, mean=x.mean(), mcse=x.std(ddof=1)/np.sqrt(R)))
e7=pd.DataFrame(rows); e7.to_csv("results/e7_convergence.csv", index=False)
print(e7.round(4).to_string(index=False)); print(f"{time.time()-t0:.0f}s total")


## Analytical checks: gap identity & mean-field vs simulation

In [ ]:
import os, time
import numpy as np, pandas as pd
from dataclasses import replace
from model import *
from manuscript import PC, PRIMARY_HORIZON
from runmode import FAST_MODE, RR, banner
banner()
os.makedirs("results", exist_ok=True); os.makedirs("figs", exist_ok=True)
# ---- gap-identity check: per-period change in G_HL vs beta_AI * Delta E[a s] + small terms ----
cfg = Config(policy="universal", T_max=60, T_post=60, patience=999)
def identity_run(seed):
    w = build_world(cfg, seed)
    ses = w["ses"]; H, L = ses==2, ses==0
    lhs, rhs = [], []
    for t in range(50):
        g0 = w["ach"][H].mean() - w["ach"][L].mean()
        as_gap = (w["a"][H]*w["s"][H]).mean() - (w["a"][L]*w["s"][L]).mean()
        T_gap = w["T"][H].mean() - w["T"][L].mean()
        al_gap = w["alpha"][H].mean() - w["alpha"][L].mean()
        step(w)
        g1 = w["ach"][H].mean() - w["ach"][L].mean()
        lhs.append(g1-g0)
        rhs.append(cfg.beta_AI*as_gap + cfg.beta_T*T_gap + cfg.beta_alpha*al_gap)
    return np.array(lhs), np.array(rhs)
L_, R_ = [], []
for r in range(RR(30)):
    l, rr = identity_run(3000+r); L_.append(l); R_.append(rr)
L_, R_ = np.array(L_), np.array(R_)
resid = (L_.mean(0) - R_.mean(0))
print(f"identity check: mean|LHS-RHS| over periods = {np.abs(resid).mean():.5f}; corr = {np.corrcoef(L_.mean(0), R_.mean(0))[0,1]:.4f}")
print(f"  cumulative gap 50 periods: LHS {L_.mean(0).sum():.4f}, RHS {R_.mean(0).sum():.4f}")
pd.DataFrame(dict(period=np.arange(50), dG=L_.mean(0), pred=R_.mean(0))).to_csv("results/identity_check.csv", index=False)
# ---- mean-field vs simulation: adoption differential on E1 grid ----
e1 = pd.read_csv("results/e1_phase.csv")
mf = pd.read_csv("results/e1_meanfield.csv", index_col=0)
sim = (e1.xH - e1.xL).values.reshape(9,9)
mfv = mf.values
print(f"mean-field vs sim adoption differential: corr={np.corrcoef(sim.ravel(), mfv.ravel())[0,1]:.3f}, MAE={np.abs(sim-mfv).mean():.3f}")
print(f"  mf range [{mfv.min():.2f},{mfv.max():.2f}] sim range [{sim.min():.2f},{sim.max():.2f}]")
pd.DataFrame(dict(sim=sim.ravel(), mf=mfv.ravel())).to_csv("results/mf_vs_sim.csv", index=False)
# ---- headline reference numbers ----
e7 = pd.read_csv("results/e7_raw.csv")
for k in ["d_d_HL","d_G_HL","d_mean_ach","d_sd","d_bg_share"]:
    x=e7[k]; se=x.std(ddof=1)/np.sqrt(len(x))
    print(f"ref universal-vs-noAI {k}: {x.mean():.4f} (MC-SE {se:.4f}) CI [{x.mean()-1.96*se:.4f},{x.mean()+1.96*se:.4f}]")
print("levels under no_ai:", e7[["d_HL_a","mean_ach_a","sd_a"]].mean().round(4).to_dict())
print("levels under universal:", e7[["d_HL_b","G_HL_b","mean_ach_b","sd_b","x_L_b","x_H_b","as_gap_b","bg_share_b"]].mean().round(4).to_dict())


## Figures F1–F7

In [ ]:
from runmode import FAST_MODE, RR, banner
banner()
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from model import *
plt.rcParams.update({"font.size":9,"figure.dpi":140,"axes.spines.top":False,"axes.spines.right":False})
COL={"L":"#D55E00","M":"#E69F00","H":"#0072B2"}

# F1: network realizations + degree balance
import networkx as nx
fig,axes=plt.subplots(1,3,figsize=(10.5,3.4))
for ax,h in zip(axes[:2],[0.0,0.9]):
    w=build_world(Config(policy="no_ai",h=h),7)
    G=nx.from_numpy_array(w["adj"]); pos=nx.spring_layout(G,seed=3,k=0.08)
    nx.draw_networkx_edges(G,pos,ax=ax,alpha=0.05,width=0.4)
    for g,name in enumerate("LMH"):
        nodes=np.flatnonzero(w["ses"]==g)
        nx.draw_networkx_nodes(G,pos,nodelist=list(nodes),node_size=6,node_color=COL[name],ax=ax,label=name)
    r=ses_assortativity(w["adj"],w["ses"]); ax.set_title(f"h={h}  (realized r={r:.2f})"); ax.axis("off")
axes[0].legend(markerscale=2,frameon=False,fontsize=8)
e0=pd.read_csv("results/e0_baseline.csv")
m=[e0[f"deg_{g}"].mean() for g in "LMH"]; s=[e0[f"deg_{g}"].std() for g in "LMH"]
axes[2].bar(["L","M","H"],m,yerr=s,color=[COL[g] for g in "LMH"],alpha=0.85)
axes[2].axhline(8,ls="--",c="k",lw=0.8); axes[2].set_ylabel("mean degree"); axes[2].set_title(f"Degree balance by SES (E0, R={len(e0)})")
axes[2].set_ylim(0,10)
plt.tight_layout(); plt.savefig("figs/F1_network_diag.png"); plt.close()

# F2: adoption trajectories by h
tr=pd.read_csv("results/e2_trajectories.csv"); tr=tr[tr.gamma_s==0.1]
R2=int(pd.read_csv("results/e2_summary.csv")["R"].iloc[0])
fig,axes=plt.subplots(1,4,figsize=(11,2.8),sharey=True)
for ax,h in zip(axes,[0.0,0.3,0.6,0.9]):
    d=tr[tr.h==h]
    for g,name in zip(["xL","xM","xH"],"LMH"):
        ax.plot(d.t,d[g],color=COL[name],label=name)
    ax.set_title(f"h={h}"); ax.set_xlabel("period"); ax.set_xlim(0,30)
axes[0].set_ylabel("adoption fraction $x_g(t)$"); axes[0].legend(frameon=False)
plt.suptitle(f"AI adoption under universal availability ($\\Delta p=0.5$); mean of {R2} replications",y=1.04,fontsize=9)
plt.tight_layout(); plt.savefig("figs/F2_adoption.png",bbox_inches="tight"); plt.close()

# F3: phase diagram + mean-field overlay
e1=pd.read_csv("results/e1_phase.csv"); H=sorted(e1.h.unique()); DP=sorted(e1.dp.unique())
Z=e1.pivot(index="h",columns="dp",values="dd_mean").values
mf=pd.read_csv("results/e1_meanfield.csv",index_col=0).values
fig,ax=plt.subplots(figsize=(5.6,4.2))
vmax=np.abs(Z).max()
pc=ax.pcolormesh(DP,H,Z,cmap="RdBu_r",vmin=-vmax,vmax=vmax,shading="nearest")
cs=ax.contour(DP,H,Z,levels=[0.25,0.5,1.0,1.5],colors="k",linewidths=0.7)
ax.clabel(cs,fmt="%.2f",fontsize=7)
cs2=ax.contour(DP,H,mf,levels=[0.10],colors="#009E73",linewidths=2,linestyles="--")
ax.plot([],[],c="#009E73",ls="--",lw=2,label="mean-field: $x_H{-}x_L=0.10$")
# cells with CI incl 0
inc=e1[(e1.lo<0)&(e1.hi>0)]
ax.scatter(inc.dp,inc.h,marker="x",c="k",s=20,label="95% CI includes 0" if len(inc) else None)
ax.set_xlabel("physical-access inequality $\\Delta p$ (mean access fixed at 0.6)")
ax.set_ylabel("network homophily $h$")
ax.set_title(f"$\\Delta d_{{HL}}$: universal AI availability vs no AI\n(9×9 grid, R={int(e1.R.iloc[0])} CRN pairs per cell)")
plt.colorbar(pc,label="$\\Delta$ standardized H–L gap"); ax.legend(frameon=False,fontsize=7,loc="upper left")
plt.tight_layout(); plt.savefig("figs/F3_phase.png"); plt.close()

# F4: achievement dynamics + identity
idc=pd.read_csv("results/identity_check.csv")
tr6=tr[tr.h==0.6]
fig,axes=plt.subplots(1,2,figsize=(9,3.2))
for g,name in zip(["aL","aM","aH"],"LMH"):
    axes[0].plot(tr6.t,tr6[g],color=COL[name],label=name)
axes[0].set_xlabel("period"); axes[0].set_ylabel("mean latent achievement"); axes[0].legend(frameon=False)
axes[0].set_title("Group achievement, universal availability (h=0.6)")
axes[1].plot(idc.period,idc.dG,lw=1.6,label="simulated $\\Delta G_{HL}(t)$")
axes[1].plot(idc.period,idc.pred,ls="--",lw=1.4,label="$\\beta_{AI}\\Delta\\mathbb{E}[a s]+$ small terms")
axes[1].set_xlabel("period"); axes[1].set_ylabel("per-period gap increment"); axes[1].legend(frameon=False,fontsize=8)
axes[1].set_title(f"Gap-identity check ({RR(30)} replications)")
plt.tight_layout(); plt.savefig("figs/F4_dynamics.png"); plt.close()

# F5: Pareto
e3=pd.read_csv("results/e3_policies.csv")
lab={"universal":"universal availability","access_full":"full access equalization","access_budget":"budget access eq. (B=60)",
     "skills":"skills support (B=60)","target_random":"random low-SES targeting","target_lowskill":"lowest-skill targeting",
     "target_degree":"degree targeting","target_betweenness":"betweenness targeting","target_bridge":"bridge targeting",
     "deseg":"desegregation (h→0.3)","combined":"combined (B=60)"}
fig,ax=plt.subplots(figsize=(6.4,4.4))
for _,r in e3.iterrows():
    ax.errorbar(r.dmean,r.dd,xerr=1.96*r.dmean_se,yerr=1.96*r.dd_se,fmt="o",ms=4,capsize=2,lw=0.8,color="#333")
    ax.annotate(lab[r.policy],(r.dmean,r.dd),fontsize=7,xytext=(4,3),textcoords="offset points")
ax.axhline(0,c="k",lw=0.6)
ax.set_xlabel("$\\Delta$ mean achievement vs no AI"); ax.set_ylabel("$\\Delta d_{HL}$ vs no AI (higher = more inequality)")
ax.set_title(f"Efficiency–equity comparison, R={RR(100)} CRN pairs; 95% MC CIs")
plt.tight_layout(); plt.savefig("figs/F5_pareto.png"); plt.close()

# F6: targeting x budget
e4=pd.read_csv("results/e4_targeting.csv")
fig,ax=plt.subplots(figsize=(5.6,3.6))
mk={"target_random":"o","target_lowskill":"s","target_degree":"^","target_betweenness":"D","target_bridge":"v"}
for pol,d in e4.groupby("policy"):
    ax.errorbar(d.B,d.dd,yerr=1.96*d.se,marker=mk[pol],ms=4,capsize=2,lw=1,label=lab[pol])
ax.set_xlabel("budget B (low-SES students equipped + seeded)"); ax.set_ylabel("$\\Delta d_{HL}$ vs no AI")
ax.set_title(f"Targeting strategies by budget (R={RR(80)})"); ax.legend(frameon=False,fontsize=7)
plt.tight_layout(); plt.savefig("figs/F6_targeting.png"); plt.close()

# F7: Sobol
sd=pd.read_csv("results/e5_sobol.csv")
x=np.arange(len(sd)); fig,ax=plt.subplots(figsize=(5.6,3.4))
ax.bar(x-0.18,sd.S1,0.36,yerr=sd.S1_conf,capsize=2,label="first-order $S_1$",color="#0072B2")
ax.bar(x+0.18,sd.ST,0.36,yerr=sd.ST_conf,capsize=2,label="total-order $S_T$",color="#D55E00")
ax.set_xticks(x); ax.set_xticklabels(list(sd.name))
ax.set_ylabel("Sobol index (response: $d_{HL}$ under universal availability)")
ax.set_title(f"Global sensitivity (Saltelli n={8 if FAST_MODE else 128}, {2 if FAST_MODE else 8} inner replications)"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig("figs/F7_sobol.png"); plt.close()
print("figures written:", __import__("os").listdir("figs"))


## Export primary outputs

In [ ]:
import shutil, json, glob
from runmode import FAST_MODE
meta = dict(base_seed=20260901, fast_mode=FAST_MODE,
            note='FAST_MODE outputs are pipeline tests only')
json.dump(meta, open('results/run_metadata.json','w'), indent=2)
shutil.make_archive('aiedu_outputs','zip',root_dir='.',base_dir='results')
shutil.make_archive('aiedu_figs','zip',root_dir='.',base_dir='figs')
print('wrote aiedu_outputs.zip, aiedu_figs.zip'); print(glob.glob('results/*.csv'))


## E8 — Confirmatory global sensitivity analysis

Seven theory-central parameters are retained a priori: `dp`, `h`, `ds0`, `theta0`, `lam`, `beta_AI`, and `sigma0`. The full run uses Sobol base sample `n=512` and eight inner stochastic replications.

In [ ]:
from runmode import FAST_MODE, banner
banner()
import os, time, numpy as np, pandas as pd
from dataclasses import replace
from model import Config, run_simulation
from manuscript import PC
from SALib.sample import sobol as sobol_sample
from SALib.analyze import sobol as sobol_analyze
os.makedirs("results_extended", exist_ok=True)
PROB7={"num_vars":7,"names":["dp","h","ds0","theta0","lam","beta_AI","sigma0"],"bounds":[[0,0.64],[0,0.9],[0,0.4],[0.1,0.3],[0,0.8],[0.004,0.020],[0.02,0.20]]}
SOBOL_N=64 if FAST_MODE else 512
INNER=2 if FAST_MODE else 8
X=sobol_sample.sample(PROB7,SOBOL_N,calc_second_order=False,seed=20260901)
Y=np.empty(len(X)); W=np.empty(len(X)); t0=time.time()
for i,theta in enumerate(X):
    kw=dict(zip(PROB7["names"],theta)); vals=[]
    for r in range(INNER):
        out=run_simulation(replace(PC(policy="universal"),**kw),9000000+10000*i+r)
        vals.append(out["d_HL"])
    Y[i]=np.mean(vals); W[i]=np.var(vals,ddof=1) if INNER>1 else 0.0
    if (i+1)%100==0 or i==len(X)-1: print(f"E8 {i+1}/{len(X)} vectors; elapsed {(time.time()-t0)/60:.1f} min")
S=sobol_analyze.analyze(PROB7,Y,calc_second_order=False,print_to_console=False,seed=20260901)
edf=pd.DataFrame({"name":PROB7["names"],"S1":S["S1"],"S1_conf":S["S1_conf"],"ST":S["ST"],"ST_conf":S["ST_conf"]})
edf["interaction_proxy"]=edf.ST-edf.S1
edf.to_csv("results_extended/e8_sobol_extended.csv",index=False)
between_var = np.var(Y, ddof=1)
noise_var_mean = np.mean(W) / INNER if INNER > 0 else np.nan
noise_ratio = noise_var_mean / between_var if between_var > 0 else np.nan
pd.DataFrame([{
    "between_theta_var": between_var,
    "mean_within_theta_var": np.mean(W),
    "within_theta_var_of_replication_mean": noise_var_mean,
    "noise_to_between_ratio": noise_ratio,
    "INNER": INNER,
    "SOBOL_N": SOBOL_N
}]).to_csv("results_extended/e8_sobol_noise_diagnostic.csv", index=False)
print("Sobol stochastic-noise ratio:", noise_ratio)
pd.DataFrame(X,columns=PROB7["names"]).assign(response=Y,within_theta_var=W).to_csv("results_extended/e8_sobol_samples.csv",index=False)
print(edf.sort_values("ST",ascending=False).round(4).to_string(index=False))


## E9 — Finite-size and network-density robustness

Checks the primary contrast over `N={250,500,1000}`, `kbar={6,8,12}`, and low/reference/high structural profiles.

In [ ]:
from runmode import FAST_MODE, banner
banner()
import os, numpy as np, pandas as pd
from model import Config, paired_contrast
from manuscript import PC
os.makedirs("results_extended",exist_ok=True)
R=6 if FAST_MODE else 30
profiles={"low":dict(h=0.3,dp=0.16),"reference":dict(h=0.6,dp=0.50),"high":dict(h=0.9,dp=0.64)}
rows=[]; counter=0
for prof,pars in profiles.items():
  for N in [250,500,1000]:
    for kbar in [6.0,8.0,12.0]:
      counter+=1
      c0=PC(N=N,C=25,kbar=kbar,policy="no_ai",**pars); c1=PC(N=N,C=25,kbar=kbar,policy="universal",**pars)
      d=paired_contrast(c0,c1,R=R,base_seed=11000000+100000*counter); x=d["d_d_HL"]; y=d["d_mean_ach"]
      se=x.std(ddof=1)/np.sqrt(R)
      rows.append(dict(profile=prof,N=N,kbar=kbar,R=R,dd_mean=x.mean(),dd_se=se,dd_lo=x.mean()-1.96*se,dd_hi=x.mean()+1.96*se,dmean_mean=y.mean(),dmean_se=y.std(ddof=1)/np.sqrt(R)))
      print(prof,N,kbar,rows[-1]["dd_mean"])
e9=pd.DataFrame(rows); e9.to_csv("results_extended/e9_size_degree_robustness.csv",index=False); display(e9)


## E10 — Seed-placement-only targeting

Isolates topology from equipment provision. The same initial physical-access environment is preserved while only low-SES seed locations change. A second environment sets full physical access before seed placement.

In [ ]:
from runmode import FAST_MODE, banner
banner()
import os, numpy as np, pandas as pd, networkx as nx
from model import Config, build_world, step, metrics, cross_frac
from manuscript import PC
os.makedirs("results_extended",exist_ok=True)
def finish_world(w):
    cfg=w["cfg"]; quiet=t=0; tstar=None
    while t<cfg.T_max:
        ch=step(w); t+=1; quiet=quiet+1 if ch==0 else 0
        if tstar is None and quiet>=cfg.patience: tstar=t-cfg.patience
        if tstar is not None and t>=tstar+cfg.T_post: break
    return metrics(w)
def choose(w,strategy,B,seed):
    elig=np.flatnonzero((w["ses"]==0)&(w["c"]==1)); B=min(B,len(elig)); rng=np.random.default_rng(seed+777777)
    if strategy=="random": return rng.choice(elig,size=B,replace=False)
    if strategy=="degree": return elig[np.argsort(-w["deg"][elig])][:B]
    if strategy=="bridge":
        cf=cross_frac(w); elig2=elig[w["deg"][elig] >= 3]
        return elig2[np.argsort(-cf[elig2])][:min(B,len(elig2))]
    if strategy=="betweenness":
        G=nx.from_numpy_array(w["adj"]); bc=nx.betweenness_centrality(G,k=min(64,w["cfg"].N),seed=int(rng.integers(1_000_000_000)))
        v=np.array([bc[i] for i in range(w["cfg"].N)])
        return elig[np.argsort(-v[elig])][:B]
    raise ValueError(strategy)
def one(seed,strategy,B,full_access=False):
    w=build_world(PC(policy="access_full" if full_access else "universal",sigma0=0.0),seed)
    t=choose(w,strategy,B,seed); w["a"][:]=0; w["a"][t]=1
    out=finish_world(w); out["n_seeded"]=len(t); return out
R=8 if FAST_MODE else 80; rows=[]
for env,full in [("access_constrained",False),("full_access",True)]:
  for B in [5,10,20,40]:
    for strategy in ["random","degree","betweenness","bridge"]:
      vals=[one(13000000+100000*B+1000*r,strategy,B,full) for r in range(R)]; d=pd.DataFrame(vals)
      rows.append(dict(environment=env,B=B,strategy=strategy,R=R,d_HL=d.d_HL.mean(),d_HL_se=d.d_HL.std(ddof=1)/np.sqrt(R),x_L=d.x_L.mean(),x_H=d.x_H.mean(),mean_ach=d.mean_ach.mean(),n_seeded=d.n_seeded.mean()))
      print(env,B,strategy,rows[-1]["d_HL"],rows[-1]["x_L"])
e10=pd.DataFrame(rows); e10.to_csv("results_extended/e10_seed_only_targeting.csv",index=False); display(e10)


## Figures F8–F10 and extended-results export

In [ ]:
from runmode import FAST_MODE, banner
banner()
import os,json,shutil,numpy as np,pandas as pd,matplotlib.pyplot as plt
os.makedirs("figs_extended",exist_ok=True)
s=pd.read_csv("results_extended/e8_sobol_extended.csv"); x=np.arange(len(s)); fig,ax=plt.subplots(figsize=(7,4)); ax.bar(x-.18,s.S1,.36,yerr=s.S1_conf,capsize=2,label="first-order S1"); ax.bar(x+.18,s.ST,.36,yerr=s.ST_conf,capsize=2,label="total-order ST"); ax.set_xticks(x); ax.set_xticklabels(["Δp","h","Δs(0)","θ0","λ","βAI","σ0"]); ax.set_ylabel("Sobol index"); ax.set_title("Confirmatory global sensitivity"); ax.legend(frameon=False); fig.tight_layout(); fig.savefig("figs_extended/F8_sobol_extended.png",dpi=220,bbox_inches="tight"); plt.close(fig)
e9=pd.read_csv("results_extended/e9_size_degree_robustness.csv")
for prof in e9.profile.unique():
 fig,ax=plt.subplots(figsize=(5.7,3.6)); d=e9[e9.profile==prof]
 for N,q in d.groupby("N"):
  q=q.sort_values("kbar"); ax.errorbar(q.kbar,q.dd_mean,yerr=1.96*q.dd_se,marker="o",capsize=2,label=f"N={N}")
 ax.set_xlabel("mean degree kbar"); ax.set_ylabel("Δd_HL"); ax.set_title(f"Size/density robustness: {prof}"); ax.legend(frameon=False); fig.tight_layout(); fig.savefig(f"figs_extended/F9_{prof}_size_degree.png",dpi=220,bbox_inches="tight"); plt.close(fig)
e10=pd.read_csv("results_extended/e10_seed_only_targeting.csv")
for env in e10.environment.unique():
 fig,ax=plt.subplots(figsize=(6,3.8)); d=e10[e10.environment==env]
 for st,q in d.groupby("strategy"):
  q=q.sort_values("B"); ax.errorbar(q.B,q.d_HL,yerr=1.96*q.d_HL_se,marker="o",capsize=2,label=st)
 ax.set_xlabel("number of low-SES seed placements"); ax.set_ylabel("final d_HL"); ax.set_title(f"Seed-placement-only: {env.replace('_',' ')}"); ax.legend(frameon=False); fig.tight_layout(); fig.savefig(f"figs_extended/F10_{env}_seed_only.png",dpi=220,bbox_inches="tight"); plt.close(fig)
json.dump({"fast_mode":FAST_MODE,"base_seed":20260901,"experiments":["E8","E9","E10"]},open("results_extended/extended_metadata.json","w"),indent=2)
shutil.make_archive("extended_results","zip",root_dir=".",base_dir="results_extended"); shutil.make_archive("extended_figures","zip",root_dir=".",base_dir="figs_extended"); print("created extended_results.zip and extended_figures.zip")


## E11 — Saturation diagnostic and non-saturating seed-placement regime

The reference threshold implies approximately
\[
\left\lceil \theta_0(1-\lambda E[T])\bar{k}\right\rceil
=\lceil 0.2(1-0.5\times0.6)8\rceil=2
\]
adopting neighbours at mean teacher support, which produces near-saturation
among access holders in the reference regime.

This experiment therefore:
1. reports the fraction of access holders who ultimately adopt, by SES and
   homophily;
2. repeats seed-placement-only targeting in the reference regime; and
3. repeats it in a pre-specified **non-saturating regime**
   `theta0=0.50`, where the mean-support threshold is approximately 3
   neighbours.

Raw paired replication-level outputs are retained so that strategy-minus-random
confidence intervals can be computed.

In [ ]:
from runmode import FAST_MODE, RR, banner
banner()
import os, numpy as np, pandas as pd, networkx as nx
from dataclasses import replace
from model import run_simulation, build_world, step, metrics, cross_frac
from manuscript import PC

os.makedirs("results_robustness", exist_ok=True)

# ---- E11a: reference adoption/access ratios ----
R_sat = RR(80)
rows = []
for h in [0.0,0.3,0.6,0.9]:
    for r in range(R_sat):
        o = run_simulation(PC(policy="universal", h=h), 21000000 + int(h*1000) + 1000*r)
        row = {"h":h, "rep":r}
        for g in "LMH":
            row[f"x_{g}"] = o[f"x_{g}"]
            row[f"acc_{g}"] = o[f"acc_{g}"]
            row[f"adopt_given_access_{g}"] = o[f"x_{g}"] / max(o[f"acc_{g}"], 1e-12)
            row[f"s_{g}"] = o[f"s_{g}"]
            row[f"ach_{g}"] = o[f"ach_{g}"]
        rows.append(row)
e11a = pd.DataFrame(rows)
e11a.to_csv("results_robustness/e11a_saturation_raw.csv", index=False)
summary = e11a.groupby("h").agg({
    **{f"adopt_given_access_{g}":"mean" for g in "LMH"},
    **{f"x_{g}":"mean" for g in "LMH"},
    **{f"acc_{g}":"mean" for g in "LMH"},
}).reset_index()
summary.to_csv("results_robustness/e11a_saturation_summary.csv", index=False)
display(summary.round(4))

# ---- helpers for seed-only experiments ----
def choose_seed_targets(world, strategy, B, seed):
    ses, c, deg = world["ses"], world["c"], world["deg"]
    eligible = np.flatnonzero((ses == 0) & (c == 1))
    if len(eligible) == 0:
        return np.array([], dtype=int)
    B = min(B, len(eligible))
    selrng = np.random.default_rng(seed + 777777)
    if strategy == "random":
        return selrng.choice(eligible, size=B, replace=False)
    if strategy == "degree":
        return eligible[np.argsort(-deg[eligible])][:B]
    if strategy == "bridge":
        cf = cross_frac(world)
        elig = eligible[deg[eligible] >= 3]
        return elig[np.argsort(-cf[elig])][:min(B,len(elig))]
    if strategy == "betweenness":
        G = nx.from_numpy_array(world["adj"])
        ksample = min(64, world["cfg"].N)
        bc = nx.betweenness_centrality(
            G, k=ksample, seed=int(selrng.integers(1_000_000_000))
        )
        bcv = np.array([bc[i] for i in range(world["cfg"].N)])
        return eligible[np.argsort(-bcv[eligible])][:B]
    raise ValueError(strategy)

def seed_only_with_times(seed, strategy, B, full_access, theta0):
    pol = "access_full" if full_access else "universal"
    cfg = PC(policy=pol, sigma0=0.0, theta0=theta0)
    w = build_world(cfg, seed)
    targets = choose_seed_targets(w, strategy, B, seed)
    w["a"][:] = 0
    w["a"][targets] = 1
    ses = w["ses"]
    hist = {g:[] for g in range(3)}
    for t in range(cfg.T_max + 1):
        for g in range(3):
            hist[g].append(w["a"][ses==g].mean())
        if t < cfg.T_max:
            step(w)
    out = metrics(w)
    for g,name in enumerate("LMH"):
        final = hist[g][-1]
        if final <= 0:
            t50 = np.nan
        else:
            t50 = next((t for t,x in enumerate(hist[g]) if x >= 0.5*final), np.nan)
        out[f"t50_{name}"] = t50
        out[f"adopt_given_access_{name}"] = out[f"x_{name}"] / max(out[f"acc_{name}"],1e-12)
    out["n_seeded"] = len(targets)
    return out

R_seed = RR(80)
budgets = [5,10,20,40]
strategies = ["random","degree","betweenness","bridge"]
regimes = [
    ("reference", 0.20),
    ("non_saturating", 0.50),
]
raw = []
for regime, theta0 in regimes:
    for env, full_access in [("access_constrained",False),("full_access",True)]:
        for B in budgets:
            for r in range(R_seed):
                seed = 22000000 + (0 if regime=="reference" else 5_000_000) + 100000*B + 1000*r
                for strategy in strategies:
                    o = seed_only_with_times(seed, strategy, B, full_access, theta0)
                    raw.append({
                        "regime":regime, "theta0":theta0, "environment":env,
                        "B":B, "rep":r, "seed":seed, "strategy":strategy,
                        **{k:o[k] for k in [
                            "d_HL","G_HL","mean_ach","x_L","x_M","x_H",
                            "acc_L","acc_M","acc_H",
                            "adopt_given_access_L","adopt_given_access_M","adopt_given_access_H",
                            "t50_L","t50_M","t50_H","n_seeded"
                        ]}
                    })
e11b = pd.DataFrame(raw)
e11b.to_csv("results_robustness/e11b_seed_only_raw.csv", index=False)

# Paired strategy-minus-random differences.
pairs = []
for keys, grp in e11b.groupby(["regime","environment","B"]):
    wide = grp.pivot(index="rep", columns="strategy", values="d_HL")
    for strategy in ["degree","betweenness","bridge"]:
        d = wide[strategy] - wide["random"]
        se = d.std(ddof=1)/np.sqrt(len(d))
        pairs.append({
            "regime":keys[0],"environment":keys[1],"B":keys[2],
            "strategy":strategy,"mean_diff_vs_random":d.mean(),
            "se":se,"lo":d.mean()-1.96*se,"hi":d.mean()+1.96*se
        })
e11pairs = pd.DataFrame(pairs)
e11pairs.to_csv("results_robustness/e11c_paired_targeting_differences.csv", index=False)
display(e11pairs.round(4))


## E12 — Scale-parameter robustness

The standardized gap is a **model-internal standardized index**, not an
empirical standardized effect size. Its denominator depends on stylized
heterogeneity and shock parameters. This factorial robustness analysis varies:

- `beta_AI/beta_alpha ∈ {0.2, 0.5, 1.2}`,
- `sigma_u ∈ {0.10, 0.15, 0.25}`,
- `sigma_eps ∈ {0.005, 0.010, 0.020}`.

It reports both `Delta d_HL` and the scale-free companion
`Delta G_HL / Delta mean achievement`.

In [ ]:
from runmode import FAST_MODE, RR, banner
banner()
import os, numpy as np, pandas as pd
from model import paired_contrast
from manuscript import PC
os.makedirs("results_robustness", exist_ok=True)

R = RR(50)
rows=[]
for ratio in [0.2,0.5,1.2]:
    for sigma_u in [0.10,0.15,0.25]:
        for sigma_eps in [0.005,0.010,0.020]:
            beta_alpha = 0.010
            beta_AI = ratio * beta_alpha
            a = PC(policy="no_ai", beta_alpha=beta_alpha, beta_AI=beta_AI,
                   sigma_u=sigma_u, sigma_eps=sigma_eps)
            b = PC(policy="universal", beta_alpha=beta_alpha, beta_AI=beta_AI,
                   sigma_u=sigma_u, sigma_eps=sigma_eps)
            d = paired_contrast(a,b,R=R,
                base_seed=24000000 + int(ratio*10000) + int(sigma_u*1000) + int(sigma_eps*100000))
            ratio_metric = d["d_G_HL"].mean()/d["d_mean_ach"].mean()
            rows.append({
                "betaAI_over_betaalpha":ratio,
                "beta_AI":beta_AI,"sigma_u":sigma_u,"sigma_eps":sigma_eps,
                "R":R,"dd":d.d_d_HL.mean(),
                "dd_se":d.d_d_HL.std(ddof=1)/np.sqrt(R),
                "dG":d.d_G_HL.mean(),
                "dmean":d.d_mean_ach.mean(),
                "gap_to_mean_gain":ratio_metric,
            })
e12=pd.DataFrame(rows)
e12.to_csv("results_robustness/e12_scale_sensitivity.csv",index=False)
display(e12.round(4))


## E13 — Access-shape and mean-access robustness

In [ ]:
from runmode import FAST_MODE, RR, banner
banner()
import os, numpy as np, pandas as pd
from model import paired_contrast
from manuscript import PC
os.makedirs("results_robustness", exist_ok=True)

R=RR(60)
rows=[]
# q sensitivity at the reference mean and access gap.
for q in [0.4,0.5,0.615,0.75]:
    a=PC(policy="no_ai",shape_mid=q,pbar=0.6,dp=0.5)
    b=PC(policy="universal",shape_mid=q,pbar=0.6,dp=0.5)
    d=paired_contrast(a,b,R=R,base_seed=25000000+int(q*10000))
    rows.append({"experiment":"q","value":q,"pbar":0.6,"dp":0.5,
                 "dd":d.d_d_HL.mean(),"se":d.d_d_HL.std(ddof=1)/np.sqrt(R),
                 "dG":d.d_G_HL.mean(),"dmean":d.d_mean_ach.mean()})

# pbar sweep at dp=0.32, which keeps all group probabilities feasible at pbar=0.8.
for pbar in [0.4,0.6,0.8]:
    a=PC(policy="no_ai",shape_mid=0.615,pbar=pbar,dp=0.32)
    b=PC(policy="universal",shape_mid=0.615,pbar=pbar,dp=0.32)
    d=paired_contrast(a,b,R=R,base_seed=25100000+int(pbar*10000))
    rows.append({"experiment":"pbar","value":pbar,"pbar":pbar,"dp":0.32,
                 "dd":d.d_d_HL.mean(),"se":d.d_d_HL.std(ddof=1)/np.sqrt(R),
                 "dG":d.d_G_HL.mean(),"dmean":d.d_mean_ach.mean()})

e13=pd.DataFrame(rows)
e13.to_csv("results_robustness/e13_access_parameter_robustness.csv",index=False)
display(e13.round(4))


## E14 — Classroom sorting and desegregation decomposition

In [ ]:
from runmode import FAST_MODE, RR, banner
banner()
import os, numpy as np, pandas as pd
from model import paired_contrast
from manuscript import PC
os.makedirs("results_robustness", exist_ok=True)

R=RR(100)
rows=[]
configs = {
    "reference":dict(h=0.6,rho=0.5),
    "rho_zero_only":dict(h=0.6,rho=0.0),
    "rho_one":dict(h=0.6,rho=1.0),
    "h_reduction_only":dict(h=0.3,rho=0.5),
    "h_and_rho_reduction":dict(h=0.3,rho=0.0),
}
for i,(name,kw) in enumerate(configs.items()):
    d=paired_contrast(PC(policy="no_ai",**kw),PC(policy="universal",**kw),
                      R=R,base_seed=26000000+100000*i)
    rows.append({"config":name,**kw,"dd":d.d_d_HL.mean(),
                 "se":d.d_d_HL.std(ddof=1)/np.sqrt(R),
                 "dG":d.d_G_HL.mean(),"dmean":d.d_mean_ach.mean()})
e14=pd.DataFrame(rows)
e14.to_csv("results_robustness/e14_classroom_deseg_decomposition.csv",index=False)
display(e14.round(4))


## E15 — Full intervention-unit sweep for target-count-constrained policies

In [ ]:
from runmode import FAST_MODE, RR, banner
banner()
import os, numpy as np, pandas as pd
from model import paired_contrast
from manuscript import PC
os.makedirs("results_robustness", exist_ok=True)

R=RR(80)
budgets=[15,30,60,90]
policies=[
    "access_budget","skills","target_random","target_lowskill",
    "target_degree","target_betweenness","target_bridge","combined"
]
rows=[]
for pi,pol in enumerate(policies):
    for B in budgets:
        d=paired_contrast(PC(policy="no_ai"),PC(policy=pol,budget=B),
                          R=R,base_seed=27000000+1_000_000*pi+10_000*B)
        rows.append({"policy":pol,"B":B,
                     "dd":d.d_d_HL.mean(),"dd_se":d.d_d_HL.std(ddof=1)/np.sqrt(R),
                     "dG":d.d_G_HL.mean(),"dmean":d.d_mean_ach.mean(),
                     "xL":d.x_L_b.mean(),"xH":d.x_H_b.mean()})
e15=pd.DataFrame(rows)
e15.to_csv("results_robustness/e15_all_policy_budget_sweep.csv",index=False)
display(e15.round(4))


## E16 — Fixed-class-size finite-size check

To isolate population size from classroom size, this companion experiment uses
20 students per classroom exactly:

- `(N,C)=(200,10)`,
- `(N,C)=(500,25)`,
- `(N,C)=(1000,50)`.

Mean network degree is fixed at 8.

In [ ]:
from runmode import FAST_MODE, RR, banner
banner()
import os, numpy as np, pandas as pd
from model import paired_contrast
from manuscript import PC
os.makedirs("results_robustness", exist_ok=True)

R=RR(40)
profiles={
    "low":dict(h=0.3,dp=0.16),
    "reference":dict(h=0.6,dp=0.50),
    "high":dict(h=0.9,dp=0.64),
}
rows=[]
for prof,kw in profiles.items():
    for N,C in [(200,10),(500,25),(1000,50)]:
        d=paired_contrast(
            PC(policy="no_ai",N=N,C=C,kbar=8.0,**kw),
            PC(policy="universal",N=N,C=C,kbar=8.0,**kw),
            R=R,base_seed=28000000+N*100+(0 if prof=="low" else 10000 if prof=="reference" else 20000)
        )
        rows.append({"profile":prof,"N":N,"C":C,"class_size":N//C,
                     "dd":d.d_d_HL.mean(),"se":d.d_d_HL.std(ddof=1)/np.sqrt(R),
                     "dG":d.d_G_HL.mean(),"dmean":d.d_mean_ach.mean()})
e16=pd.DataFrame(rows)
e16.to_csv("results_robustness/e16_fixed_class_size.csv",index=False)
display(e16.round(4))


## E17 — Three-group reference outcomes

In [ ]:
from runmode import FAST_MODE, RR, banner
banner()
import os, numpy as np, pandas as pd
from model import run_simulation
from manuscript import PC
os.makedirs("results_robustness", exist_ok=True)

R=RR(200)
rows=[]
for r in range(R):
    o=run_simulation(PC(policy="universal"),29000000+1000*r)
    rows.append({
        "rep":r,
        **{f"acc_{g}":o[f"acc_{g}"] for g in "LMH"},
        **{f"x_{g}":o[f"x_{g}"] for g in "LMH"},
        **{f"s_{g}":o[f"s_{g}"] for g in "LMH"},
        **{f"ach_{g}":o[f"ach_{g}"] for g in "LMH"},
        "d_HL":o["d_HL"],"G_HL":o["G_HL"],"mean_ach":o["mean_ach"]
    })
e17=pd.DataFrame(rows)
e17.to_csv("results_robustness/e17_three_group_reference_raw.csv",index=False)
summ=[]
for g in "LMH":
    summ.append({
        "SES":g,
        "access":e17[f"acc_{g}"].mean(),
        "adoption":e17[f"x_{g}"].mean(),
        "adoption_given_access":(e17[f"x_{g}"]/e17[f"acc_{g}"].clip(lower=1e-12)).mean(),
        "skill":e17[f"s_{g}"].mean(),
        "achievement":e17[f"ach_{g}"].mean(),
    })
e17s=pd.DataFrame(summ)
e17s.to_csv("results_robustness/e17_three_group_reference_summary.csv",index=False)
display(e17s.round(4))


## Combined results export

In [ ]:
from runmode import FAST_MODE
from manuscript import PRIMARY_HORIZON
import os, sys, json, shutil, importlib.metadata as im, pandas as pd

os.makedirs("results_robustness",exist_ok=True)

# Copy all manuscript-relevant analysis tables into one package.
copy_files = [
    "results/e0_baseline.csv",
    "results/e0_summary.csv",
    "results/e1_phase.csv",
    "results/e1b_channels.csv",
    "results/e1c_dp_ds0.csv",
    "results/e2_summary.csv",
    "results/e2_trajectories.csv",
    "results/e3_policies.csv",
    "results/e3_raw.csv",
    "results/e4_targeting.csv",
    "results/e5_morris.csv",
    "results/e5_sobol.csv",
    "results/e6_robustness.csv",
    "results/e7_raw.csv",
    "results/e7_convergence.csv",
    "results/identity_check.csv",
    "results/mf_vs_sim.csv",
    "results_extended/e8_sobol_extended.csv",
    "results_extended/e8_sobol_samples.csv",
    "results_extended/e8_sobol_noise_diagnostic.csv",
    "results_extended/e9_size_degree_robustness.csv",
    "results_extended/e10_seed_only_targeting.csv",
]
for src in copy_files:
    if os.path.exists(src):
        dest = "results_robustness/" + src.replace("/", "__")
        shutil.copy2(src, dest)

versions={}
for pkg in ["numpy","scipy","pandas","networkx","matplotlib","SALib"]:
    try:
        versions[pkg]=im.version(pkg)
    except Exception:
        versions[pkg]="unavailable"

meta={
    "fast_mode":FAST_MODE,
    "base_seed":20260901,
    "primary_horizon_periods":PRIMARY_HORIZON,
    "python":sys.version.split()[0],
    "packages":versions,
    "non_saturating_regime":{"theta0":0.50,"seed_design":"low-SES seed-only"},
    "note":"Only fast_mode=false outputs are manuscript-grade."
}
json.dump(meta,open("results_robustness/full_run_metadata.json","w"),indent=2)

shutil.make_archive("all_results","zip",root_dir=".",base_dir="results_robustness")
print("Created all_results.zip")
print(json.dumps(meta,indent=2))


## Horizon and threshold follow-up experiments

These two experiments do **not** alter the common 50-period primary experiments.

- E18: structural-channel decomposition at a shorter horizon `T_h=20`
  to quantify the horizon dependence of the skill channel.
- E19: intermediate seed-only threshold `theta0=0.40` under full physical
  access at `B={20,40}`, testing whether the targeting reversal is confined
  to the stringent `theta0=0.50` regime.

E18–E20 use `ProcessPoolExecutor`; on Colab this runs on all available cores.

In [ ]:
import os
from runmode import RR, banner
banner()
import numpy as np, pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import replace
from model import Config, run_simulation
OUT='round2_targeted_results'; os.makedirs(OUT,exist_ok=True)
configs = {
 'all_channels': dict(dp=0.5, ds0=0.3, h=0.6),
 'equal_access': dict(dp=0.0, ds0=0.3, h=0.6),
 'equal_skills': dict(dp=0.5, ds0=0.0, h=0.6),
 'no_homophily': dict(dp=0.5, ds0=0.3, h=0.0),
 'access_only': dict(dp=0.5, ds0=0.0, h=0.0),
 'skills_only': dict(dp=0.0, ds0=0.3, h=0.0),
 'no_inequality': dict(dp=0.0, ds0=0.0, h=0.6),
 'pure_null': dict(dp=0.0, ds0=0.0, h=0.0),
}
def HC(policy,kw):
    return Config(policy=policy,T_max=20,T_post=20,patience=21,**kw)
def task(args):
    name,kw,r,seed=args
    a=run_simulation(HC('no_ai',kw),seed); b=run_simulation(HC('universal',kw),seed)
    return name,r,b['d_HL']-a['d_HL'],b['G_HL']-a['G_HL'],b['mean_ach']-a['mean_ach'],b['x_L'],b['x_H']
R=RR(50)
tasks=[]
for ii,(name,kw) in enumerate(configs.items()):
    for r in range(R):
        tasks.append((name,kw,r,33000000+ii*100000+r*1000))
rows=[]
with ProcessPoolExecutor(max_workers=5) as ex:
    for res in ex.map(task,tasks,chunksize=2):
        rows.append(res)
raw=pd.DataFrame(rows,columns=['config','rep','dd','dG','dmean','xL','xH'])
raw.to_csv(f'{OUT}/e18_channels_T20_raw.csv',index=False)
out=[]
for name,g in raw.groupby('config'):
    se=g.dd.std(ddof=1)/np.sqrt(len(g))
    out.append(dict(horizon=20,config=name,R=len(g),dd=g.dd.mean(),se=se,lo=g.dd.mean()-1.96*se,hi=g.dd.mean()+1.96*se,dG=g.dG.mean(),dmean=g.dmean.mean(),xL=g.xL.mean(),xH=g.xH.mean()))
s=pd.DataFrame(out); s.to_csv(f'{OUT}/e18_channels_T20.csv',index=False)
print(s[['config','R','dd','se','dG','dmean']].sort_values('config').round(4).to_string(index=False))


In [ ]:
import os
from runmode import RR, banner
banner()
import numpy as np, pandas as pd, networkx as nx
from concurrent.futures import ProcessPoolExecutor
from model import Config, build_world, step, metrics, cross_frac
OUT='round2_targeted_results'; os.makedirs(OUT,exist_ok=True)

def choose(world,strategy,B,seed):
    ses,c,deg=world['ses'],world['c'],world['deg']
    elig=np.flatnonzero((ses==0)&(c==1)); B=min(B,len(elig))
    rng=np.random.default_rng(seed+777777)
    if strategy=='random': return rng.choice(elig,size=B,replace=False)
    if strategy=='degree': return elig[np.argsort(-deg[elig])][:B]
    if strategy=='bridge':
        cf=cross_frac(world); e2=elig[deg[elig]>=3]
        return e2[np.argsort(-cf[e2])][:min(B,len(e2))]
    if strategy=='betweenness':
        G=nx.from_numpy_array(world['adj']); ksample=min(64,world['cfg'].N)
        bc=nx.betweenness_centrality(G,k=ksample,seed=int(rng.integers(1_000_000_000)))
        bcv=np.array([bc[i] for i in range(world['cfg'].N)])
        return elig[np.argsort(-bcv[elig])][:B]
    raise ValueError(strategy)

def task(args):
    B,r,strategy=args; seed=34000000+B*100000+r*1000
    cfg=Config(policy='access_full',sigma0=0.0,theta0=0.4,T_max=50,T_post=50,patience=51)
    w=build_world(cfg,seed); t=choose(w,strategy,B,seed); w['a'][:]=0; w['a'][t]=1
    for _ in range(50): step(w)
    o=metrics(w)
    return B,r,seed,strategy,o['d_HL'],o['G_HL'],o['mean_ach'],o['x_L'],o['x_M'],o['x_H']
R=RR(50)
tasks=[(B,r,s) for B in [20,40] for r in range(R) for s in ['random','degree','betweenness','bridge']]
with ProcessPoolExecutor(max_workers=5) as ex:
    rows=list(ex.map(task,tasks,chunksize=1))
raw=pd.DataFrame(rows,columns=['B','rep','seed','strategy','d_HL','G_HL','mean_ach','x_L','x_M','x_H'])
raw.to_csv(f'{OUT}/e19_theta04_seed_only_raw.csv',index=False)
s=raw.groupby(['B','strategy']).agg(d_HL=('d_HL','mean'),d_HL_se=('d_HL',lambda x:x.std(ddof=1)/np.sqrt(len(x))),x_L=('x_L','mean'),x_M=('x_M','mean'),x_H=('x_H','mean'),mean_ach=('mean_ach','mean')).reset_index()
s.to_csv(f'{OUT}/e19_theta04_seed_only_summary.csv',index=False)
pairs=[]
for B,g in raw.groupby('B'):
    wide=g.pivot(index='rep',columns='strategy',values='d_HL')
    for st in ['degree','betweenness','bridge']:
        for metric,d in [('signed',wide[st]-wide['random']),('absolute',wide[st].abs()-wide['random'].abs())]:
            se=d.std(ddof=1)/np.sqrt(len(d)); pairs.append(dict(B=B,strategy=st,metric=metric,mean=d.mean(),se=se,lo=d.mean()-1.96*se,hi=d.mean()+1.96*se))
p=pd.DataFrame(pairs); p.to_csv(f'{OUT}/e19_theta04_paired.csv',index=False)
print('SUMMARY')
print(s.round(4).to_string(index=False))
print('\nPAIRED')
print(p.round(4).to_string(index=False))


## E20 — Fine threshold/seed sweep and mean-field comparison

Full physical access, random/degree/betweenness/bridge low-SES seed placement, $B\in\{20,40\}$, $\theta_0=0.20,0.25,\ldots,0.60$, 50 replications. For random low-SES seeding, a seed-specific block mean-field iteration is evaluated on the same threshold grid.

In [ ]:
import os
from runmode import RR, banner
banner(), copy
import numpy as np, pandas as pd, networkx as nx
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import binom
from model import Config, build_world, step, metrics, cross_frac

OUT='final_theta_sweep_results'; os.makedirs(OUT,exist_ok=True)
THETAS=np.round(np.arange(0.20,0.6001,0.05),2)
BS=[20,40]
STRATS=['random','degree','betweenness','bridge']
R=RR(50)

def choose_all(w,B,seed):
    ses,c,deg=w['ses'],w['c'],w['deg']
    elig=np.flatnonzero((ses==0)&(c==1)); B=min(B,len(elig))
    rng=np.random.default_rng(seed+777777)
    out={}
    out['random']=rng.choice(elig,size=B,replace=False)
    out['degree']=elig[np.argsort(-deg[elig])][:B]
    cf=cross_frac(w); e2=elig[deg[elig]>=3]
    out['bridge']=e2[np.argsort(-cf[e2])][:min(B,len(e2))]
    G=nx.from_numpy_array(w['adj']); ksample=min(64,w['cfg'].N)
    bc=nx.betweenness_centrality(G,k=ksample,seed=int(rng.integers(1_000_000_000)))
    bcv=np.array([bc[i] for i in range(w['cfg'].N)])
    out['betweenness']=elig[np.argsort(-bcv[elig])][:B]
    return out

def task(args):
    theta,B,r=args
    seed=36000000 + int(round(theta*1000))*100000 + B*1000 + r
    cfg=Config(policy='access_full',sigma0=0.0,theta0=float(theta),T_max=50,T_post=50,patience=51)
    w=build_world(cfg,seed)
    targets=choose_all(w,B,seed)
    rows=[]
    for s in STRATS:
        wc=copy.deepcopy(w)
        wc['a'][:]=0; wc['a'][targets[s]]=1
        for _ in range(50): step(wc)
        o=metrics(wc)
        rows.append((theta,B,r,seed,s,o['d_HL'],o['G_HL'],o['mean_ach'],o['x_L'],o['x_M'],o['x_H']))
    return rows

def neighbor_matrix(cfg):
    N=cfg.N; n=np.array(cfg.frac)*N
    omega0=(n-1)/(N-1); omega=omega0+cfg.h*(1-omega0)
    M=np.zeros((3,3))
    for g in range(3):
        M[g,g]=omega[g]
        others=[gg for gg in range(3) if gg!=g]
        tot=sum(n[gg] for gg in others)
        for gg in others:
            M[g,gg]=(1-omega[g])*n[gg]/tot
    return M,n

def meanfield_seedonly(theta,B,iters=500,tol=1e-12):
    cfg=Config(policy='access_full',sigma0=0.0,theta0=float(theta))
    M,n=neighbor_matrix(cfg)
    q=np.array([B/n[0],0.0,0.0])
    x=q.copy()
    thbar=cfg.theta0*(1-cfg.lam*cfg.T_mu); k=int(round(cfg.kbar)); m=int(np.ceil(thbar*k))
    for it in range(iters):
        w=np.clip(M@x,0,1)
        tail=1-binom.cdf(m-1,k,w)
        xn=q+(1-q)*tail
        if np.max(np.abs(xn-x))<tol:
            x=xn; break
        x=xn
    return x,m,it+1

if __name__=='__main__':
    tasks=[(float(th),B,r) for th in THETAS for B in BS for r in range(R)]
    allrows=[]
    with ProcessPoolExecutor(max_workers=6) as ex:
        for pack in ex.map(task,tasks,chunksize=1):
            allrows.extend(pack)
    raw=pd.DataFrame(allrows,columns=['theta0','B','rep','seed','strategy','d_HL','G_HL','mean_ach','x_L','x_M','x_H'])
    raw.to_csv(f'{OUT}/theta_sweep_raw.csv',index=False)
    summ=raw.groupby(['theta0','B','strategy']).agg(
        d_HL=('d_HL','mean'),d_HL_se=('d_HL',lambda x:x.std(ddof=1)/np.sqrt(len(x))),
        x_L=('x_L','mean'),x_L_se=('x_L',lambda x:x.std(ddof=1)/np.sqrt(len(x))),
        x_M=('x_M','mean'),x_M_se=('x_M',lambda x:x.std(ddof=1)/np.sqrt(len(x))),
        x_H=('x_H','mean'),x_H_se=('x_H',lambda x:x.std(ddof=1)/np.sqrt(len(x))),
        mean_ach=('mean_ach','mean')
    ).reset_index()
    summ.to_csv(f'{OUT}/theta_sweep_summary.csv',index=False)
    mf=[]
    for th in THETAS:
        for B in BS:
            x,m,nit=meanfield_seedonly(float(th),B)
            mf.append({'theta0':th,'B':B,'threshold_count':m,'mf_x_L':x[0],'mf_x_M':x[1],'mf_x_H':x[2],'iterations':nit})
    mf=pd.DataFrame(mf); mf.to_csv(f'{OUT}/theta_sweep_meanfield.csv',index=False)
    ran=summ[summ.strategy=='random'].merge(mf,on=['theta0','B'])
    checks=[]
    for B,g in ran.groupby('B'):
        for G in ['L','M','H']:
            d=g[f'x_{G}']-g[f'mf_x_{G}']
            checks.append({'B':B,'SES':G,'MAE':np.abs(d).mean(),'RMSE':np.sqrt((d*d).mean()),'r':g[f'x_{G}'].corr(g[f'mf_x_{G}'])})
    checkdf=pd.DataFrame(checks); checkdf.to_csv(f'{OUT}/theta_sweep_mf_validation.csv',index=False)
    print('SUMMARY random/degree selected')
    print(summ[summ.strategy.isin(['random','degree'])].round(4).to_string(index=False))
    print('\nMEANFIELD')
    print(mf.round(4).to_string(index=False))
    print('\nVALIDATION')
    print(checkdf.round(4).to_string(index=False))
